# Avaliação do Modelo — Score de Risco de Inadimplência

Avaliação do modelo selecionado em `train.py`, com:
1. **Comparação dos 3 algoritmos** (LogisticRegression, RandomForest, GradientBoosting) pela **CV-AUC** (critério de seleção);
2. **Performance do vencedor no holdout** (ROC, KS, matriz de confusão);
3. **Análise de threshold** (trade-off entre capturar inadimplência e volume de aprovações);
4. **Interpretabilidade** (importância das features).

> Decisão: os 3 candidatos são comparados pela **AUC-ROC média em validação cruzada (CV-AUC)**,
> calculada só no treino — o holdout fica **intocado** durante a seleção. Só o **vencedor** é
> avaliado no holdout (estimativa final honesta).
> Métrica principal: **AUC-ROC**; complementares: **KS** e **Recall da classe inadimplente**.

In [ ]:
import json
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, f1_score)

import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

import sys, os
# Sobe para a raiz do projeto: os caminhos em config.py (Model/..., Dados/...) são
# relativos a ela. Funciona tanto rodando de dentro de Model/ quanto já na raiz.
while not (os.path.isdir('Model') and os.path.isdir('Dados')):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        break  # chegou na raiz do disco sem achar: deixa o erro estourar adiante
    os.chdir(parent)
sys.path.insert(0, os.path.join(os.getcwd(), 'Model'))  # p/ achar Model/config.py
import config

## 1. Comparação dos modelos (metrics.json)

In [ ]:
with open(config.METRICS_PATH, 'r', encoding='utf-8') as f:
    metrics = json.load(f)

# A seleção do modelo é feita pela AUC-ROC média em validação cruzada (CV-AUC), calculada
# SÓ no treino. É o critério real de escolha — por isso a comparação dos 3 é feita por ele.
# As métricas de holdout (auc_roc/ks/accuracy...) só existem para o VENCEDOR, que é o único
# avaliado no conjunto de teste (seções 2-5); os perdedores têm apenas cv_auc.
comp = pd.DataFrame(metrics['results']).T.sort_values('cv_auc', ascending=False)
print("Melhor modelo (por CV-AUC):", metrics['best_model'])
display(comp[['cv_auc']].style.format('{:.4f}').background_gradient(cmap='Greens'))

ax = comp['cv_auc'].plot(kind='bar', figsize=(9, 5), color='#2ca25f')
plt.title('Comparação dos modelos — CV-AUC (critério de seleção)')
plt.ylabel('AUC-ROC média (validação cruzada)'); plt.xticks(rotation=15)
plt.axhline(0.5, color='gray', ls='--', lw=1, label='acaso (0,5)'); plt.legend()
plt.show()

# --- Quadro-resumo do vencedor (métricas de holdout já gravadas no metrics.json pelo train.py) ---
# Exibe só as MÉTRICAS DE DECISÃO do projeto: AUC-ROC, KS e Recall da classe inadimplente (+ a
# CV-AUC da seleção). Acurácia e precisão são propositalmente OMITIDAS aqui: numa base desbalanceada
# (~8% de calote) elas enganam numa leitura rápida (paradoxo da acurácia; precisão naturalmente
# baixa) — aparecem, COM a devida explicação e contexto, na seção 4 (matriz de confusão).
best = metrics['results'][metrics['best_model']]
resumo = pd.Series({
    'CV-AUC (seleção, treino)':   best['cv_auc'],
    'AUC-ROC (holdout)':          best['auc_roc'],
    'KS (holdout)':               best['ks'],
    'Recall — inadimplente':      best['recall_default'],
})
print(f"\nQuadro do vencedor ({metrics['best_model']}) — hiperparâmetros afinados: {best.get('best_params')}")
print("(acurácia e precisão: ver seção 4 — enganam fora de contexto na base desbalanceada)")
display(resumo.to_frame('valor').style.format('{:.4f}').background_gradient(cmap='Blues'))

> 📊 **Como interpretar:** a tabela compara os 3 algoritmos pela **CV-AUC** — a AUC-ROC média
> em validação cruzada (`StratifiedKFold`, 5 folds) calculada **só no treino**. Este é o **critério
> de seleção**: vence quem separa melhor bons e maus pagadores de forma **consistente** entre os
> folds, sem nunca "olhar" o holdout. O **GradientBoosting** tem a maior CV-AUC (~0,76), à frente
> da LogisticRegression e da RandomForest — por isso foi o escolhido.
>
> A **AUC** responde: *pegando ao acaso um cliente que deu calote e um que pagou, qual a chance de
> o modelo dar nota de risco maior ao que deu calote?* **1,0** = acerta sempre (separação perfeita);
> **0,5 = acerta metade, igual a jogar uma moeda** (o modelo não distingue quem paga de quem não
> paga). O nosso ~0,76 significa acertar essa ordenação ~76% das vezes — bem acima do acaso.
>
> **Por que CV-AUC e não a AUC no holdout para comparar?** Selecionar o modelo olhando o holdout
> seria "espiar a prova" — o holdout deixaria de ser uma estimativa honesta de dados nunca vistos.
> A CV-AUC roda inteiramente **dentro do treino** (5 folds, média das 5 medições), então a
> comparação é justa e o holdout fica reservado para a **avaliação final** de um único modelo.
> Por isso só o **vencedor** tem métricas de holdout — resumidas no **quadro do vencedor** acima e
> detalhadas ao vivo nas **seções 2 a 5**. Os perdedores param na seleção por CV.
>
> **Nota sobre o quadro do vencedor:** ele mostra só as **métricas de decisão** (AUC-ROC, KS, Recall).
> **Acurácia** e **precisão** ficam de fora *de propósito*: numa base desbalanceada (~8% de calote)
> elas enganam numa leitura rápida (paradoxo da acurácia; precisão naturalmente baixa). Elas aparecem,
> **com a explicação completa**, na **seção 4** (matriz de confusão).

## 2. Carregar o melhor modelo e reproduzir o holdout

In [ ]:
abt = pd.read_parquet(config.ABT_PATH)
drop_cols = [c for c in (config.ID_COL, config.TARGET_COL) if c in abt.columns]
X = abt.drop(columns=drop_cols)
y = abt[config.TARGET_COL]

# Mesmo split do treino (mesmo random_state) -> reproduz o conjunto de teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE, stratify=y)

with open(config.MODEL_PATH, 'rb') as f:
    bundle = pickle.load(f)
model = bundle['model']
print("Modelo carregado:", bundle['model_name'])

y_proba = model.predict_proba(X_test)[:, 1]
print(f"AUC-ROC no teste: {roc_auc_score(y_test, y_proba):.4f}")

## 3. Curva ROC e estatística KS

In [ ]:
fpr, tpr, thr = roc_curve(y_test, y_proba)
ks = np.max(tpr - fpr)
ks_idx = np.argmax(tpr - fpr)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_test, y_proba):.3f}')
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_title('Curva ROC'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].legend()

axes[1].plot(thr, tpr, label='TPR (sensibilidade)')
axes[1].plot(thr, fpr, label='FPR')
axes[1].axvline(thr[ks_idx], color='red', ls='--', label=f'KS = {ks:.3f} @ thr={thr[ks_idx]:.2f}')
axes[1].set_title('Estatística KS'); axes[1].set_xlabel('Threshold'); axes[1].legend()
axes[1].set_xlim(0, 1)
plt.tight_layout(); plt.show()

> 📊 **Como interpretar:** na **curva ROC** (esquerda), quanto mais a curva sobe para o canto superior
> esquerdo, melhor; a área sob ela é a AUC (~0,77). A **linha diagonal tracejada é o modelo aleatório**
> (AUC 0,5) — quanto mais a curva fica **acima** dela, mais longe do "chute" o modelo está.
>
> **O que é a estatística KS (Kolmogorov-Smirnov):** é a medida clássica de **poder de separação** em
> crédito. Imagine ordenar todos os clientes pela nota de risco do modelo e ir somando, conforme o corte
> se desloca, **duas curvas acumuladas**: a dos **inadimplentes** já capturados (TPR, sensibilidade) e a
> dos **bons clientes** marcados por engano (FPR). O **KS é a maior distância vertical entre essas duas
> curvas** (a linha vermelha no gráfico da direita, ~0,41).
>
> Em outras palavras: no ponto de corte ideal, o modelo já separou **41 pontos percentuais a mais** de
> inadimplentes do que de bons pagadores. Quanto **maior** o KS, mais o modelo "afasta" os dois grupos —
> idealmente jogando os inadimplentes para as notas altas e os bons para as baixas.
>
> **Régua de leitura (mercado de crédito):** **0** = não separa nada (curvas coladas, modelo aleatório);
> **< 0,20** fraco; **~0,20–0,30** aceitável; **~0,30–0,50** bom; **> 0,50** excelente (raro). O nosso
> **~0,41** é um resultado **bom** para risco de crédito. Diferença para a AUC: a AUC resume a separação
> em **todos** os cortes possíveis num só número; o KS aponta o **melhor corte** e o tamanho da separação
> exatamente ali — por isso costuma orientar a escolha do threshold de aprovação.

## 4. Matriz de confusão (threshold padrão = 0.5)

In [ ]:
y_pred = (y_proba >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)
# values_format='d' evita a notação científica (ex.: 4e+04) que trunca os números grandes no gráfico.
ConfusionMatrixDisplay(cm, display_labels=['Adimplente', 'Inadimplente']).plot(
    cmap='Blues', values_format='d')
plt.title('Matriz de confusão (threshold 0.5)')
plt.show()

tn, fp, fn, tp = cm.ravel()

# Métricas globais no threshold 0.5
acuracia   = accuracy_score(y_test, y_pred)              # % de acertos no total
precisao   = precision_score(y_test, y_pred)             # dos marcados como inadimplentes, quantos são
recall     = recall_score(y_test, y_pred)                # dos inadimplentes reais, quantos o modelo pega
# Baseline ingênuo: "aprovar todo mundo" (nunca prever inadimplência)
acc_ingenua = (y_test == 0).mean()

print(f"Falsos Negativos (inadimplência aprovada): {fn}  <-- o erro mais caro para o negócio")
print(f"Falsos Positivos (bom cliente negado): {fp}")
print()
print(f"Acurácia (acertos totais):              {acuracia:.2%}")
print(f"Precisão (classe inadimplente):         {precisao:.2%}")
print(f"Recall (inadimplentes capturados):      {recall:.2%}")
print()
print(f"[Referência] Acurácia de um modelo que aprova TODOS: {acc_ingenua:.2%}")
print("  -> em base desbalanceada (~8% de calote) a acurácia sozinha ENGANA:")
print("     o modelo burro 'aprova todos' tem acurácia MAIOR, mas recall = 0% (não pega nenhum calote).")

> 📊 **Como interpretar:** os acertos ficam na diagonal. Os erros são de dois tipos: **Falsos Negativos**
> (inadimplentes que o modelo aprovaria — o erro mais caro) e **Falsos Positivos** (bons clientes negados).
> No corte padrão (0,5), o modelo **captura ~69% dos inadimplentes**. A maior parte dos erros é *negar bom
> cliente*, não *aprovar calote* — alinhado ao objetivo de reduzir prejuízo.
>
> ### Acurácia e Precisão — os números estão bons ou ruins?
>
> | Métrica | Valor (thr 0,5) | O que mede | Avaliação neste projeto |
> |---|---|---|---|
> | **Acurácia** | **~71%** | % de acertos no total (adimplentes + inadimplentes) | ⚠️ **Enganosa aqui** — *não* é o foco |
> | **Precisão** (classe 1) | **~17%** | Dos que o modelo marca como inadimplentes, quantos de fato são | 🟡 **Baixa, mas esperada** |
> | **Recall** (classe 1) | **~69%** | Dos inadimplentes reais, quantos o modelo pega | ✅ **Bom** — é o que queremos maximizar |
>
> **Por que a acurácia de ~71% NÃO é um bom termômetro aqui (paradoxo da acurácia):** a base é
> **desbalanceada** (~8% de inadimplentes). Um modelo **burro que aprovasse todos** os clientes acertaria
> **~92%** dos casos — acurácia *maior* que a do nosso modelo! — mas teria **recall de 0%**: não pegaria
> **nenhum** calote. Ou seja, alta acurácia pode esconder um modelo inútil para o problema. Nosso modelo
> usa `class_weight='balanced'` e **abre mão de acurácia de propósito** (71% < 92%) para **capturar
> inadimplência** (recall 69%). Por isso a acurácia entra só como contexto, e a **AUC-ROC/KS/recall** são
> as métricas de decisão.
>
> **Por que a precisão de ~17% é baixa mas aceitável:** com apenas ~8% de calote na população, ao baixar a
> régua para pegar mais inadimplentes o modelo inevitavelmente marca muitos bons pagadores por engano
> (Falsos Positivos) — o que **derruba a precisão**. Isso é o esperado e **desejável** dado o objetivo de
> negócio: **negar um bom cliente é bem mais barato que aprovar um calote**. A precisão é a alavanca que
> se troca por recall ao mexer no **threshold** (seção 5): subir o corte aumenta a precisão e reduz o
> recall, e vice-versa.

## 5. Análise de threshold (trade-off de negócio)

In [ ]:
rows = []
for t in np.arange(0.1, 0.91, 0.1):
    pred = (y_proba >= t).astype(int)
    rows.append({
        'threshold': round(t, 2),
        'recall_default': recall_score(y_test, pred, zero_division=0),
        'precision_default': precision_score(y_test, pred, zero_division=0),
        'f1_default': f1_score(y_test, pred, zero_division=0),
        'taxa_aprovacao': (pred == 0).mean(),
    })
thr_df = pd.DataFrame(rows)
display(thr_df.style.format({'recall_default': '{:.2%}', 'precision_default': '{:.2%}',
                             'f1_default': '{:.2%}', 'taxa_aprovacao': '{:.2%}'}))

plt.plot(thr_df['threshold'], thr_df['recall_default'], marker='o', label='Recall (inadimplentes capturados)')
plt.plot(thr_df['threshold'], thr_df['taxa_aprovacao'], marker='s', label='Taxa de aprovação')
plt.xlabel('Threshold'); plt.title('Trade-off: capturar inadimplência vs. aprovar crédito')
plt.legend(); plt.show()

> 📊 **Como interpretar (decisão de negócio):** baixar o threshold faz o **recall subir** (pega mais
> inadimplentes), mas derruba a **taxa de aprovação** (nega mais gente); subir faz o contrário. Não há valor
> "certo" — depende do apetite de risco. Ex.: a **0,3** o modelo pega ~91% dos inadimplentes mas aprova só
> ~36%; a **0,7** aprova ~90% mas pega só ~37%. O 0,5 é só o ponto de partida. A coluna **`f1_default`**
> resume o equilíbrio recall × precision em cada corte: ela é **maior nos thresholds intermediários** (onde
> os dois ficam mais parelhos) e cai nos extremos — útil como referência, mas a decisão final segue o
> apetite de risco, não o pico do F1.

## 6. Interpretabilidade — importância das features

Avaliamos a importância por **três métodos independentes** que se validam entre si:
**(6a) importância nativa** (impureza das árvores), **(6b) permutation importance** (queda na AUC ao
embaralhar cada feature) e **(7) SHAP** (contribuição por cliente).

In [ ]:
clf = model.named_steps['clf'] if hasattr(model, 'named_steps') else model
if hasattr(clf, 'feature_importances_'):
    imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False).head(20)
    plt.figure(figsize=(10, 8))
    sns.barplot(x=imp.values, y=imp.index, palette='viridis')
    plt.title('Top 20 features mais importantes'); plt.xlabel('Importância')
    plt.show()
else:
    # Modelos lineares: usa o valor absoluto dos coeficientes
    imp = pd.Series(np.abs(clf.coef_[0]), index=X.columns).sort_values(ascending=False).head(20)
    sns.barplot(x=imp.values, y=imp.index, palette='viridis'); plt.title('Top 20 coeficientes (|peso|)'); plt.show()

> 📊 **Como interpretar:** as features no topo são as que mais pesam na decisão. Dominam os **scores
> externos** (`EXT_SOURCE_*`), seguidos de idade (`DAYS_BIRTH`), das razões financeiras e das agregações de
> histórico (`BUREAU_*`/`PREV_*`) — coerente com a EDA e com o domínio de crédito.

In [ ]:
from sklearn.inspection import permutation_importance

# Permutation importance: embaralha cada coluna e mede a QUEDA na AUC-ROC do modelo no holdout.
# Model-agnostic e medida na métrica que importa (AUC) -> corrige o viés da importância por impureza,
# que infla features contínuas / de alta cardinalidade e as correlacionadas (caso dos EXT_SOURCE_*).
# Amostra do holdout + n_repeats baixo p/ rodar rápido (reajusta previsões, não re-treina).
perm_sample = X_test.sample(min(5000, len(X_test)), random_state=config.RANDOM_STATE)
perm = permutation_importance(
    model, perm_sample, y_test.loc[perm_sample.index],
    scoring='roc_auc', n_repeats=5, random_state=config.RANDOM_STATE, n_jobs=-1)

perm_imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
perm_std = pd.Series(perm.importances_std, index=X.columns)
top = perm_imp.head(20)

# barh com barra de erro (desvio entre as repetições); invertido p/ a maior ficar no topo
plt.figure(figsize=(10, 8))
plt.barh(top.index[::-1], top.values[::-1],
         xerr=perm_std.loc[top.index].values[::-1], color='#8c2d81')
plt.title('Top 20 features por Permutation Importance (queda na AUC-ROC)')
plt.xlabel('Queda média na AUC ao embaralhar a feature')
plt.tight_layout(); plt.show()

print("Top 10 (queda na AUC ± desvio):")
for f in top.head(10).index:
    print(f"  {f:<28} {perm_imp[f]:.4f} ± {perm_std[f]:.4f}")

> 📊 **Permutation importance — por que somamos este segundo método:** a importância nativa do gráfico
> anterior vem da **impureza** (quanto cada feature reduz o erro nos *splits* das árvores). É rápida, mas
> tem um viés conhecido: **infla features contínuas / de alta cardinalidade** e distribui mal o crédito
> entre variáveis **correlacionadas** — exatamente o nosso caso (`EXT_SOURCE_*` contínuas e várias razões
> financeiras correlacionadas). A **permutation importance** ataca isso por outro ângulo: para cada
> feature, ela **embaralha os valores daquela coluna** no holdout e mede **quanto a AUC-ROC cai**. Se
> embaralhar quase não muda a AUC, a feature é pouco usada; se derruba muito, ela é essencial.
>
> Vantagens: (1) é **model-agnostic** (mede o modelo já treinado, não a estrutura interna); (2) mede o
> impacto na **métrica que de fato importa** (AUC); (3) roda sobre o **holdout**, refletindo o poder
> preditivo **fora da amostra**. O resultado **confirma** o ranking anterior — os `EXT_SOURCE_*` seguem no
> topo —, o que dá **robustez** à leitura de importância (dois métodos independentes + SHAP concordam).
> Nota de leitura: features **muito correlacionadas** podem ter a permutation importance *subestimada*
> (o modelo compensa a coluna embaralhada usando a "irmã" correlacionada) — por isso olhamos os três
> métodos em conjunto, não isoladamente. Custo: reajusta previsões `n_features × n_repeats` vezes; por
> isso rodamos numa amostra do holdout com `n_repeats=5`.

## 7. Interpretabilidade com SHAP

SHAP atribui a cada feature a sua contribuição para a previsão de cada cliente (em log-odds).
O *beeswarm* abaixo resume essas contribuições na amostra: à direita = empurra para
**inadimplência**; à esquerda = empurra para **adimplência**. É a mesma técnica usada no endpoint
`/explain` da API e na explicação do app Streamlit.

In [ ]:
# SHAP confirma a contribuição de cada feature. Executado em amostra para ser rápido.
try:
    import shap
    sample = X_test.sample(min(500, len(X_test)), random_state=config.RANDOM_STATE)
    explainer = shap.TreeExplainer(clf)   # GradientBoosting -> TreeExplainer (exato e rápido)
    shap_values = explainer(sample)
    shap.plots.beeswarm(shap_values, max_display=15, show=True)
except ImportError:
    print('SHAP não instalado neste ambiente (pip install shap).')

> 📊 **Como ler o beeswarm:** cada ponto é um cliente; à **direita** a feature empurra a previsão para
> **inadimplência**, à **esquerda** para **adimplência**. A cor é o valor da feature (vermelho = alto,
> azul = baixo). Ex.: valores **baixos** de `EXT_SOURCE` (azul) aparecem à direita → score externo ruim
> **aumenta** o risco. Confirma, cliente a cliente, o que a importância de features mostrou no agregado.

## Conclusões da avaliação
- **GradientBoosting** lidera em AUC (~0.773) e KS (~0.41), superando LogisticRegression e RandomForest.
- O modelo **captura ~69% dos inadimplentes** no threshold 0.5; a precisão (~17%) baixa é esperada pelo
  desbalanceamento (~8%) e é aceitável dado o foco em **reduzir Falsos Negativos (inadimplência)**.
- **Acurácia ~71% NÃO é o termômetro aqui** (paradoxo da acurácia): um modelo que aprovasse todos teria
  ~92% de acurácia e recall 0%. O modelo troca acurácia por recall de propósito (`class_weight='balanced'`);
  a decisão segue AUC-ROC/KS/recall.
- O **threshold é uma alavanca de negócio**: baixá-lo captura mais inadimplentes (maior recall) ao
  custo de negar mais bons clientes (menor taxa de aprovação). A escolha final depende do apetite de risco.
- As **features mais importantes** são os scores externos (`EXT_SOURCE_*`), idade (`DAYS_BIRTH`),
  as razões financeiras e as **agregações de histórico de crédito** (`BUREAU_*`/`PREV_*`) — coerente
  com a EDA e com o domínio de crédito.
- **Enriquecimento com histórico** (`bureau` + `previous_application`): elevou o AUC do GradientBoosting
  de **0.766 → 0.773** (KS 0.395 → 0.407). Ganho modesto e consistente; os `EXT_SOURCE_*` já
  concentravam grande parte do sinal.
- **Controle de overfitting**: árvores rasas (`max_depth` baixo) e `min_samples_leaf` alto na
  RandomForest; `learning_rate`/`n_estimators` moderados no GradientBoosting; avaliação em holdout.